## Step 1: Setup
Import libraries and connect to the Anthropic API using a secure environment variable.

In [1]:
import anthropic #the official libary to talk to claude's API 
import os #lets python interact directly with the host operating system 
from dotenv import load_dotenv #lets python read the content of the .env file 

load_dotenv('/Users/samrithisatish/Desktop/transcript-summarizer/.env') #this reads the .env file and loads the anthrotopic-api-key to memory so that python can access it

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY")) #opening a phone line to claude's API
print("Client ready ")

Client ready 


## Step 2: Load & Clean Transcript
Read the transcript file and remove blank lines and whitespace.

In [2]:
def clean_transcript(text): 
    lines = text.split('\n') #takes the raw text and splits it into individual lines wherever there is a line break 
    cleaned = []
    for line in lines:
        line = line.strip()
        if line == '':
            continue
        cleaned.append(line) #goes through every line, removes extra spaces from the edges, and skips any blank lines. Only keeps lines that have actual content 
    return '\n'.join(cleaned) #joins all the kept lines together back together into one clean block of text. 


with open('transcript.txt', 'r') as f: #reads the file 
    raw_text = f.read()

cleaned_text = clean_transcript(raw_text)
print(f"Transcript loaded: {len(cleaned_text.split())} words") #shows how many words were found 
print("\nFirst 200 characters preview:")
print(cleaned_text[:200]) 

Transcript loaded: 597 words

First 200 characters preview:
ACME TECHNOLOGIES Q3 2024 EARNINGS CALL TRANSCRIPT
Date: October 15, 2024
Participants: CEO John Marshall, CFO Sarah Chen, Analyst Mike Torres (Goldman Sachs), Analyst Priya Patel (Morgan Stanley)
OPE


## Step 3: Chunk Text
Split the transcript into overlapping chunks to handle long documents within LLM context limits.

In [3]:
def chunk_text(text, chunk_size=2000, overlap=100): # splits the text into different chunks (2000 words) and each chunk has a 100 word overlap with the next 
    words = text.split() #splits the entire transcript into a list of individual words. 
    chunks = []
    start = 0 #sets starting postion to word 0 
    
    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)
        start = end - overlap 
        '''each interation it takes words from start to end, joins them as a string, and saves it as a chunk,
        then moves the start forward. '''
    
    return chunks

chunks = chunk_text(cleaned_text)
print(f"Total chunks created: {len(chunks)}") #how many chunks created 
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk.split())} words") #how many words in each chunk  

Total chunks created: 1
Chunk 1: 597 words


## Step 4: Summarize Chunks
Send each chunk to Claude and collect structured summaries.

In [4]:
def summarize_chunk(chunk, chunk_number): #chunk= piece of text and chunk_number= which chunk it is for tracking
    print(f"Summarizing chunk {chunk_number}...")
    
    message = client.messages.create( #sends request to claude (calls the claude API)
        model="claude-haiku-4-5-20251001", #specifies which claude model 
        max_tokens=500, #claude's response should be under 500 tokens(375 words)
        messages=[ #actual prompt given to claude 
            {
                "role": "user", #means the prompt is given by me 
                "content": "You are analyzing a business earnings call transcript.\nSummarize this section and extract:\n1. Key points discussed\n2. Important numbers or metrics mentioned\n3. Any decisions or forward looking statements\nSection:\n" + chunk
            }
        ]
    )
    return message.content[0].text #extracts the text from claude's response and returns it from the function

chunk_summaries = [] #to store summaries that come back from claude 
for i, chunk in enumerate(chunks): #enumerate gives you both the index and chunk text itself 
    summary = summarize_chunk(chunk, i+1)
    chunk_summaries.append(summary)
    print(f"Chunk {i+1} done\n")
    print(summary)
    print("-" * 50) #printing of output 

Summarizing chunk 1...
Chunk 1 done

# ACME Technologies Q3 2024 Earnings Call Summary

## 1. KEY POINTS DISCUSSED

**Operational Highlights:**
- Strong quarter driven by cloud division growth and successful AI platform launch
- AcmeAI platform exceeded internal targets by 35% with rapid enterprise adoption
- Strong adoption in financial services and healthcare verticals
- Hardware segment facing supply chain headwinds expected to continue into Q4
- Significant hiring focused on AI/machine learning talent

**Financial Performance:**
- Gross margin expansion reflecting shift to higher-margin cloud/software revenue
- Strong free cash flow generation supporting capital allocation flexibility
- Net income and EPS growth exceeded prior year period

**Competitive Position:**
- Differentiation through vertical-specific AI models (particularly healthcare with FDA clearance)
- No meaningful pricing pressure from competitors observed yet

## 2. IMPORTANT NUMBERS & METRICS

| Metric | Q3 2024 | P

## Step 5: Final Report
Combine all chunk summaries into one structured intelligence brief.

In [5]:
def final_summary(summaries):
    combined = '\n\n'.join(summaries)
    
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,
        messages=[
            {
                "role": "user",
                "content": "Combine these summaries into one clean final report with these sections:\n- Executive Summary\n- Key Financial Metrics\n- Strategic Highlights\n- Risks & Headwinds\n- Outlook\n\nSummaries:\n\n" + combined
            }
        ]
    )
    return message.content[0].text

result = final_summary(chunk_summaries)
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(result) 

FINAL SUMMARY
# ACME Technologies Q3 2024 Earnings Report

## Executive Summary

ACME Technologies delivered a strong third quarter, driven by exceptional momentum in its cloud and AI businesses. The company achieved 23% year-over-year revenue growth to $847M, with net income rising 32% to $94M. The flagship AcmeAI platform exceeded adoption targets by 35%, onboarding 1,200 enterprise customers in its first 90 days. While hardware segment headwinds persist, the strategic shift toward higher-margin cloud and software solutions is delivering measurable financial benefits and positioning the company for sustainable growth.

## Key Financial Metrics

| Metric | Q3 2024 | Prior Period | Change |
|--------|---------|--------------|--------|
| **Total Revenue** | $847M | $689M | +23% YoY |
| **Cloud Division Revenue** | $491M | 41% YoY growth | 58% of total |
| **Gross Margin** | 68% | 63% | +500 bps |
| **Operating Expenses** | $312M | $265M | +18% YoY |
| **Net Income** | $94M | $71M | +32%

## Step 6: Sentiment Analysis
Classify the overall tone of the transcript.

In [6]:
def analyze_sentiment(text):
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": "Analyze the overall sentiment of this business transcript. Respond with:\n- Overall sentiment: (Positive/Neutral/Negative)\n- Confidence: (High/Medium/Low)\n- One sentence explanation\n\nTranscript:\n" + text
            }
        ]
    )
    return message.content[0].text

sentiment_result = analyze_sentiment(cleaned_text)
print("SENTIMENT ANALYSIS")
print("=" * 40)
print(sentiment_result)

SENTIMENT ANALYSIS
# Sentiment Analysis

**Overall sentiment:** Positive

**Confidence:** High

**Explanation:** ACME Technologies demonstrated strong financial performance with 23% revenue growth, exceptional AI platform adoption (35% above targets), improved margins, raised full-year guidance, and robust free cash flow, which substantially outweighs the disclosed hardware segment headwinds.


## Step 7: Theme Extraction
Extract the top key themes discussed in the transcript.

In [7]:
def extract_themes(text):
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": "Extract the top 5-7 key themes from this business transcript. Return them as a numbered list, each theme in 3-5 words.\n\nTranscript:\n" + text
            }
        ]
    )
    return message.content[0].text

themes_result = extract_themes(cleaned_text)
print("KEY THEMES")
print("=" * 40)
print(themes_result)

KEY THEMES
# Top Key Themes

1. Strong cloud and AI growth momentum

2. Supply chain pressures impacting hardware segment

3. Significant AI talent acquisition and investment

4. Improved profitability and margin expansion

5. Strategic M&A in data infrastructure space

6. Competitive differentiation through vertical-specific solutions


## Step 8: Named Entity Recognition
Extract people, companies, products, and financial figures.

In [8]:
def extract_entities(text):
    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[
            {
                "role": "user",
                "content": "Extract named entities from this business transcript. Return them in these categories:\n- People mentioned\n- Companies mentioned\n- Products mentioned\n- Key financial figures\n\nTranscript:\n" + text
            }
        ]
    )
    return message.content[0].text

entities_result = extract_entities(cleaned_text)
print("NAMED ENTITIES")
print("=" * 40)
print(entities_result)

NAMED ENTITIES
# Named Entities Extracted from ACME Technologies Q3 2024 Earnings Call

## People Mentioned
- John Marshall (CEO, ACME Technologies)
- Sarah Chen (CFO, ACME Technologies)
- Mike Torres (Analyst, Goldman Sachs)
- Priya Patel (Analyst, Morgan Stanley)

## Companies Mentioned
- ACME Technologies
- Goldman Sachs
- Morgan Stanley
- Salesforce
- Microsoft

## Products Mentioned
- AcmeAI platform
- AI product suite
- Cloud division
- Hardware segment
- Healthcare AI (with FDA clearance)

## Key Financial Figures
- **Q3 2024 Total Revenue:** $847 million
- **Year-over-Year Growth:** 23%
- **Cloud Division Growth:** 41%
- **Cloud Division % of Total Revenue:** 58%
- **AcmeAI Enterprise Customers Onboarded (90 days):** 1,200
- **Exceeded Internal Targets:** 35%
- **Hardware Segment Decline:** 12% (quarter over quarter)
- **New Employees Added (Q3):** 340
- **Total Global Employees:** 4,200+
- **Gross Margin:** 68% (up from 63% in


## Step 9: JSON Export
Store all outputs as structured JSON for downstream use.

In [ ]:
import json
#stores all the outputs as key value pairs in a dictionary 
#JSON format makes the data usable by dashboards, databases, or any downstream system

intelligence_brief = {
    "transcript_summary": result,
    "sentiment": sentiment_result,
    "key_themes": themes_result,
    "named_entities": entities_result
}

#save to disk as a .json file 
with open('intelligence_brief.json', 'w') as f:
    json.dump(intelligence_brief, f, indent=2)

print("JSON saved successfully ")
print("\nPreview:")
print(json.dumps(intelligence_brief, indent=2)[:500])

JSON saved successfully 

Preview:
{
  "transcript_summary": "# ACME Technologies Q3 2024 Earnings Report\n\n## Executive Summary\n\nACME Technologies delivered a strong third quarter, driven by exceptional momentum in its cloud and AI businesses. The company achieved 23% year-over-year revenue growth to $847M, with net income rising 32% to $94M. The flagship AcmeAI platform exceeded adoption targets by 35%, onboarding 1,200 enterprise customers in its first 90 days. While hardware segment headwinds persist, the strategic shift t


## Step 10: PDF Export
Generate a professional PDF intelligence brief.

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch

#PDF is the most professional and shareable format for business users
#converts the JSON data into a clean, readable brief

def export_pdf(brief, filename="intelligence_brief.pdf"):
    doc = SimpleDocTemplate(filename, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    #title
    story.append(Paragraph("Transcript Intelligence Brief", styles['Title']))
    story.append(Spacer(1, 0.2 * inch))

    #each section
    sections = {
        "Executive Summary": brief["transcript_summary"],
        "Sentiment Analysis": brief["sentiment"],
        "Key Themes": brief["key_themes"],
        "Named Entities": brief["named_entities"]
    }

    for heading, content in sections.items():
        story.append(Paragraph(heading, styles['Heading1']))
        story.append(Spacer(1, 0.1 * inch))
        # Clean content for PDF
        clean = content.replace('#', '').replace('**', '').replace('*', '')
        for line in clean.split('\n'):
            if line.strip():
                story.append(Paragraph(line.strip(), styles['Normal']))
                story.append(Spacer(1, 0.05 * inch))
        story.append(Spacer(1, 0.2 * inch))

    doc.build(story)
    print(f"PDF saved as {filename} ")

export_pdf(intelligence_brief)

PDF saved as intelligence_brief.pdf 
